In [3]:
from simple_cnn import SimpleCNN
import torch
import torch.nn.functional as F
import utils

In [29]:
activation = {}
def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

In [30]:
random_input = torch.randn(1, 3, 32, 32)
random_weights = torch.randn(5, 3, 3, 3)

In [31]:
model = SimpleCNN()
model.conv1.register_forward_hook(get_activation("conv1"))

output = model(random_input)


In [32]:
output

tensor([[-0.6532, -0.3162,  0.1549,  1.0947,  0.4260,  0.8298,  0.1983, -0.4084,
          0.4834, -0.9071]], grad_fn=<AddmmBackward0>)

In [33]:
print(activation["conv1"].shape)  # Should print the shape of the output from conv1
print(activation["conv1"])

torch.Size([1, 32, 32, 32])
tensor([[[[-1.0708e-01, -6.8811e-01,  8.7034e-02,  ..., -1.9819e-01,
            6.6489e-01, -2.5404e-01],
          [ 5.8969e-02, -5.1790e-01, -9.9202e-01,  ...,  8.9838e-02,
           -3.1393e-01, -3.0021e-01],
          [-7.0311e-01,  3.9156e-02, -1.4252e+00,  ..., -5.1365e-01,
            1.2218e-01, -4.5605e-01],
          ...,
          [ 2.8760e-01,  8.8108e-01,  3.2680e-01,  ..., -5.8066e-01,
            9.3569e-01, -4.4874e-01],
          [ 6.2549e-01,  5.9794e-01,  5.9148e-01,  ...,  8.6230e-02,
           -1.6170e-01,  4.8258e-01],
          [-6.8897e-01,  1.3915e-01, -2.6559e-01,  ..., -6.5966e-01,
           -2.3685e-01, -4.0280e-01]],

         [[ 3.4438e-01,  7.2677e-03, -2.1874e-01,  ..., -2.9919e-01,
           -6.3698e-01,  4.1300e-01],
          [-6.4986e-01, -1.2589e-01,  2.4298e-01,  ..., -3.3763e-01,
            1.3272e-01, -6.6031e-01],
          [-7.7426e-02, -3.7389e-01,  1.4572e+00,  ...,  4.5259e-01,
            7.1778e-01,  9.311

In [ ]:
def cross_correlation(input: torch.Tensor, kernel: torch.Tensor):
    kh, kw = kernel.shape
    ih, iw = input.shape
    H_out, W_out = ih - kh + 1, iw - kw + 1

    output = input.new_zeros(H_out, W_out)

    for i in range(H_out):
        for j in range(W_out):
            acc = 0
            for p in range(kh):
                for q in range(kw):
                    acc += input[i + p][j + q] * kernel[p][q]

            output[i][j] = acc

    return output


In [35]:
x = torch.ones(7, 7)
w = torch.full((3, 3), 0.5)

In [36]:
y = cross_correlation(x, w)
truth = torch.full((5, 5), 4.5)
assert(torch.allclose(y, truth))

In [37]:
# following formula at https://docs.pytorch.org/docs/2.15/generated/torch.nn.Conv2d.html

def conv2d(input: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:
    # input is N,C_in,H,W
    # weight is C_out,C_in,kh,kw

    if not (input.ndim == 4 and weight.ndim == 4):
        raise ValueError("Tensors must be 4D")

    N, C_in, H, W = input.shape
    print(input.shape)
    print(weight.shape)
    assert(C_in == weight.shape[1]) 
    C_out, _, kh, kw = weight.shape
    H_out = H - kh + 1
    W_out = W - kw + 1

    output = torch.zeros(N, C_out, H_out, W_out)
    print("Output shape:", output.shape)

    for i in range(N):
        for j in range(C_out):
            acc = input.new_zeros(H_out, W_out)
            for k in range(C_in):
                # input[i, k] is W x H (one input image channel)
                # weight[j, k] is kw x kh (one filter)
                acc += cross_correlation(input[i, k], weight[j, k])

            output[i, j] = acc 

    return output

In [38]:
my_output = conv2d(random_input, random_weights)
ref = F.conv2d(random_input, random_weights)
utils.tensors_equal(my_output, ref)

torch.Size([1, 3, 32, 32])
torch.Size([5, 3, 3, 3])
Output shape: torch.Size([1, 5, 30, 30])
max abs diff: 3.814697265625e-06


True

In [39]:
my_output

tensor([[[[ -3.0905,  -8.4856,  -6.0250,  ...,  -4.6804,   2.3622,   1.4005],
          [  6.5111,   6.2295,  -3.2139,  ...,  -1.8489,   0.9557,  10.9414],
          [ -1.3003,  -1.1467,  -3.8449,  ...,   0.4646,   6.4168,  -8.0720],
          ...,
          [  4.2325,  -1.8026,  -0.4850,  ...,   3.8672,  -0.7233,   1.0044],
          [  0.3383,  -3.4223,  -4.4180,  ...,  -0.4183,  -3.5182,   6.3467],
          [ -4.5935,  -5.8829,   4.9585,  ...,   0.5562,  -0.1166,  -4.9473]],

         [[ -3.6985,   3.1596,   5.8386,  ...,  -0.4567,   3.1769, -12.7956],
          [ -5.4762,  11.2317,  -1.5539,  ...,  -4.5904,   2.6687,   6.9140],
          [ -0.4820,  -8.2149,  13.4978,  ...,   1.7552, -10.5741,  -0.9942],
          ...,
          [ -6.1918,   3.5515,  -8.5565,  ...,  -0.8245,  -5.1177,  10.8456],
          [  0.9852,  -8.8330,  -1.7143,  ...,   5.4503,  -1.9332,   8.2045],
          [  0.8684,   2.8299,   9.2160,  ...,  -6.0241,   7.5371,   3.7728]],

         [[  2.8701,  10.0859,

In [1]:
def maxPool2d(input: torch.Tensor, kernel_size: int, stride: int | None = None) -> torch.Tensor:
    # input is N,C,H,W
    if not (input.ndim == 4):
        raise ValueError("Input tensor must be 4D")

    # torch's MaxPool2d defaults stride to kernel_size
    stride = kernel_size if stride is None else stride

    N, C, H, W = input.shape
    kh, kw = kernel_size, kernel_size

    H_out = (H - kh) // stride + 1
    W_out = (W - kw) // stride + 1

    output = input.new_zeros(N, C, H_out, W_out)

    for i in range(N):
        for j in range(C):
            for h in range(H_out):
                for w in range(W_out):
                    window = input[i, j, h * stride : h * stride + kh, w * stride : w * stride + kw]
                    output[i, j, h, w] = window.max()

    return output

In [4]:
for shape, k in [((1, 3, 32, 32), 2), ((2, 5, 15, 15), 2), ((1, 1, 7, 7), 3)]:
    x = torch.randn(shape)
    mine = maxPool2d(x, k)
    ref = F.max_pool2d(x, k)  # default stride = kernel_size
    assert mine.shape == ref.shape, (mine.shape, ref.shape)
    assert torch.allclose(mine, ref)

# explicit stride=1 still matches
x = torch.randn(1, 2, 9, 9)
assert torch.allclose(maxPool2d(x, 2, stride=1), F.max_pool2d(x, 2, stride=1))

# dtype/device follow the input, like conv2d
assert maxPool2d(torch.randn(1, 1, 8, 8, dtype=torch.float64), 2).dtype == torch.float64

print("maxPool2d matches torch")

maxPool2d matches torch
